In [1]:

%pip install xgboost


  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [3]:
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import xgboost as xgb

In [4]:
# ── Load data ──
# If your files are in a different folder, update these paths
TEST_PATH       = "data/test.csv"
SUBMISSION_PATH = "data/submission_format.csv"

test = pd.read_csv(TEST_PATH)
submission_format = pd.read_csv(SUBMISSION_PATH)

print(f"✅ test.csv loaded       → {test.shape[0]:,} rows × {test.shape[1]} columns")
print(f"✅ submission_format.csv → {submission_format.shape[0]:,} rows")
print(f"\nColumns in test.csv:\n{test.columns.tolist()}")
print(f"\nFirst 3 rows:\n{test.head(3)}")


✅ test.csv loaded       → 381,952 rows × 15 columns
✅ submission_format.csv → 47,744 rows

Columns in test.csv:
['time', 'channel', 'PRN', 'Carrier_Doppler_hz', 'Pseudorange_m', 'RX_time', 'TOW', 'Carrier_phase', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD', 'CN0']

First 3 rows:
     time channel  PRN  Carrier_Doppler_hz  Pseudorange_m    RX_time  \
0  111402     ch0    8         4749.068417   4.841489e+06  263154.82   
1  111402     ch1    9         1995.777378   2.449848e+06  263154.82   
2  111402     ch2   27         3458.024120   3.738822e+06  263154.82   

             TOW  Carrier_phase             EC             LC             PC  \
0  263154.803851 -435396.111594  101998.429688  100788.812500  111415.289062   
1  263154.811828 -201479.950180  100812.039062   98424.367188  109174.476562   
2  263154.807529 -356000.635312  138335.140625  125640.570312  138446.281250   

             PIP           PQP          TCD        CN0  
0 -107731.039062 -28414.615234  4660.467773  43.559540  
1 

In [5]:
# ─────────────────────────────────────────────
# PHASE 1 — CLEAN DATA
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("PHASE 1: Cleaning data...")
print("="*60)

# Some channels are "inactive" (PRN=0, all values zero).
# These are not real satellite readings — we drop them BEFORE
# feature engineering, then handle them at aggregation time.

total_rows   = len(test)
inactive     = (test['PRN'] == 0).sum()
active_test  = test[test['PRN'] != 0].copy()

print(f"Total rows         : {total_rows:,}")
print(f"Inactive (PRN=0)   : {inactive:,}  ({100*inactive/total_rows:.1f}%)")
print(f"Active rows kept   : {len(active_test):,}")

# Sanity check — no NaN values expected
print(f"\nNull values in active data:\n{active_test.isnull().sum()}")




PHASE 1: Cleaning data...
Total rows         : 381,952
Inactive (PRN=0)   : 194,130  (50.8%)
Active rows kept   : 187,822

Null values in active data:
time                  0
channel               0
PRN                   0
Carrier_Doppler_hz    0
Pseudorange_m         0
RX_time               0
TOW                   0
Carrier_phase         0
EC                    0
LC                    0
PC                    0
PIP                   0
PQP                   0
TCD                   0
CN0                   0
dtype: int64


In [6]:
# ─────────────────────────────────────────────
# PHASE 2 — FEATURE ENGINEERING
# ─────────────────────────────────────────────
def engineer_features(df):
    """
    Groups by timestamp and builds one feature row per time step.
    These features capture both signal quality and cross-channel behaviour.
    """
    g = df.groupby('time')

    feats = pd.DataFrame(index=g.groups.keys())
    feats.index.name = 'time'

    # ── Basic signal features ──────────────────────────────────
    # Mean and spread of key signals across channels at each timestamp
    for col in ['Carrier_Doppler_hz', 'Pseudorange_m', 'CN0',
                'Carrier_phase', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD']:
        feats[f'{col}_mean'] = g[col].mean()
        feats[f'{col}_std']  = g[col].std().fillna(0)
        feats[f'{col}_max']  = g[col].max()
        feats[f'{col}_min']  = g[col].min()
        feats[f'{col}_range']= g[col].max() - g[col].min()

    # ── Number of active channels ──────────────────────────────
    # Spoofing can cause channels to drop out suddenly
    feats['active_channels'] = g['PRN'].count()

    # ── Doppler spoofing indicators ────────────────────────────
    # Real satellites all move differently → large Doppler spread
    # A spoofer broadcasts same signal to all → suspiciously low std
    feats['doppler_cv'] = (
        feats['Carrier_Doppler_hz_std'] /
        (feats['Carrier_Doppler_hz_mean'].abs() + 1e-9)
    )  # Coefficient of variation

    # ── CN0 (signal strength) indicators ──────────────────────
    # Spoofers often produce unnaturally high, uniform signal strength
    feats['cn0_cv']      = feats['CN0_std'] / (feats['CN0_mean'] + 1e-9)
    feats['cn0_high']    = (feats['CN0_mean'] > 50).astype(int)  # Suspiciously strong

    # ── Pseudorange indicators ─────────────────────────────────
    feats['range_spread'] = feats['Pseudorange_m_std']  # Low spread = suspicious

    # ── Temporal (time-series) features ───────────────────────
    # Sort by time so .diff() gives actual consecutive differences
    feats = feats.sort_index()

    # How much did each signal JUMP since last timestamp?
    feats['doppler_jump']    = feats['Carrier_Doppler_hz_mean'].diff().abs().fillna(0)
    feats['pseudorange_jump']= feats['Pseudorange_m_mean'].diff().abs().fillna(0)
    feats['cn0_jump']        = feats['CN0_mean'].diff().abs().fillna(0)
    feats['phase_jump']      = feats['Carrier_phase_mean'].diff().abs().fillna(0)

    # Rolling statistics over last 10 time steps (detect sustained anomaly)
    feats['doppler_roll_std']    = feats['Carrier_Doppler_hz_mean'].rolling(10, min_periods=1).std().fillna(0)
    feats['pseudorange_roll_std']= feats['Pseudorange_m_mean'].rolling(10, min_periods=1).std().fillna(0)
    feats['cn0_roll_mean']       = feats['CN0_mean'].rolling(10, min_periods=1).mean().fillna(0)

    # ── TCD (Time Clock Drift) features ───────────────────────
    # TCD should correlate strongly with Doppler in real signals
    feats['tcd_doppler_ratio'] = (
        feats['TCD_mean'] / (feats['Carrier_Doppler_hz_mean'] + 1e-9)
    )

    feats = feats.reset_index()
    return feats


# Run feature engineering on ACTIVE rows only
features = engineer_features(active_test)

# Some timestamps had ALL channels inactive (PRN=0) → they won't appear.
# We need predictions for ALL timestamps in submission_format.
# Fill missing timestamps with 0 features (they'll be predicted as normal).
all_times  = pd.DataFrame({'time': submission_format['time']})
features   = all_times.merge(features, on='time', how='left').fillna(0)

print(f"✅ Feature matrix shape: {features.shape}")
print(f"   ({features.shape[0]:,} timestamps × {features.shape[1]-1} features)")
print(f"\nFeature columns:\n{[c for c in features.columns if c != 'time']}")

✅ Feature matrix shape: (47744, 64)
   (47,744 timestamps × 63 features)

Feature columns:
['Carrier_Doppler_hz_mean', 'Carrier_Doppler_hz_std', 'Carrier_Doppler_hz_max', 'Carrier_Doppler_hz_min', 'Carrier_Doppler_hz_range', 'Pseudorange_m_mean', 'Pseudorange_m_std', 'Pseudorange_m_max', 'Pseudorange_m_min', 'Pseudorange_m_range', 'CN0_mean', 'CN0_std', 'CN0_max', 'CN0_min', 'CN0_range', 'Carrier_phase_mean', 'Carrier_phase_std', 'Carrier_phase_max', 'Carrier_phase_min', 'Carrier_phase_range', 'EC_mean', 'EC_std', 'EC_max', 'EC_min', 'EC_range', 'LC_mean', 'LC_std', 'LC_max', 'LC_min', 'LC_range', 'PC_mean', 'PC_std', 'PC_max', 'PC_min', 'PC_range', 'PIP_mean', 'PIP_std', 'PIP_max', 'PIP_min', 'PIP_range', 'PQP_mean', 'PQP_std', 'PQP_max', 'PQP_min', 'PQP_range', 'TCD_mean', 'TCD_std', 'TCD_max', 'TCD_min', 'TCD_range', 'active_channels', 'doppler_cv', 'cn0_cv', 'cn0_high', 'range_spread', 'doppler_jump', 'pseudorange_jump', 'cn0_jump', 'phase_jump', 'doppler_roll_std', 'pseudorange_ro

In [7]:
# ─────────────────────────────────────────────
# PHASE 3 — ISOLATION FOREST → PSEUDO-LABELS
# ─────────────────────────────────────────────
feature_cols = [c for c in features.columns if c != 'time']
X = features[feature_cols].values

# Scale features so no single feature dominates due to different units
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Running Isolation Forest... (may take ~60 seconds)")
iso_forest = IsolationForest(
    n_estimators=300,      # Number of trees — more = more stable
    contamination=0.15,    # Assumed fraction of spoofed timestamps
    max_samples='auto',
    random_state=42,
    n_jobs=-1              # Use all CPU cores
)
iso_forest.fit(X_scaled)

# Predictions: -1 = anomaly (spoofed), +1 = normal
iso_raw    = iso_forest.predict(X_scaled)         # -1 or +1
iso_scores = iso_forest.score_samples(X_scaled)   # Lower = more anomalous

# Convert: -1 → label 1 (spoofed), +1 → label 0 (normal)
pseudo_labels = (iso_raw == -1).astype(int)

n_spoofed = pseudo_labels.sum()
n_normal  = (pseudo_labels == 0).sum()
print(f"\n✅ Pseudo-labels generated:")
print(f"   Normal  (0): {n_normal:,}  ({100*n_normal/len(pseudo_labels):.1f}%)")
print(f"   Spoofed (1): {n_spoofed:,} ({100*n_spoofed/len(pseudo_labels):.1f}%)")


Running Isolation Forest... (may take ~60 seconds)

✅ Pseudo-labels generated:
   Normal  (0): 40,582  (85.0%)
   Spoofed (1): 7,162 (15.0%)


In [8]:

# ─────────────────────────────────────────────
# PHASE 4 — TRAIN XGBoost ON PSEUDO-LABELS
# ─────────────────────────────────────────────
# 80/20 split — stratified to keep class ratio the same in both splits
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, pseudo_labels,
    test_size=0.2,
    random_state=42,
    stratify=pseudo_labels
)

print(f"Training set  : {X_train.shape[0]:,} samples")
print(f"Validation set: {X_val.shape[0]:,} samples")

# Handle class imbalance — tell XGBoost to pay more attention to rare class
# scale_pos_weight = (number of negatives) / (number of positives)
scale = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"Class imbalance ratio (scale_pos_weight): {scale:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale,   # Handles class imbalance
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

# Train with early stopping — stops if validation loss doesn't improve
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50  # Print progress every 50 rounds
)

# Evaluate on validation set
y_val_pred = xgb_model.predict(X_val)
val_f1 = f1_score(y_val, y_val_pred, average='weighted')
print(f"\n✅ Validation Weighted F1: {val_f1:.4f}")
print(f"\nDetailed classification report:\n")
print(classification_report(y_val, y_val_pred, target_names=['Normal', 'Spoofed']))


Training set  : 38,195 samples
Validation set: 9,549 samples
Class imbalance ratio (scale_pos_weight): 5.67
[0]	validation_0-logloss:0.65356
[50]	validation_0-logloss:0.11735
[100]	validation_0-logloss:0.06531
[150]	validation_0-logloss:0.04906
[200]	validation_0-logloss:0.04143
[250]	validation_0-logloss:0.03701
[300]	validation_0-logloss:0.03395
[350]	validation_0-logloss:0.03185
[400]	validation_0-logloss:0.03064
[450]	validation_0-logloss:0.02974
[499]	validation_0-logloss:0.02900

✅ Validation Weighted F1: 0.9871

Detailed classification report:

              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99      8117
     Spoofed       0.95      0.97      0.96      1432

    accuracy                           0.99      9549
   macro avg       0.97      0.98      0.97      9549
weighted avg       0.99      0.99      0.99      9549



In [9]:
# ─────────────────────────────────────────────
# PHASE 5 — THRESHOLD TUNING FOR BEST F1
# ─────────────────────────────────────────────
# Get probabilities (probability of being class 1 = spoofed)
y_val_proba = xgb_model.predict_proba(X_val)[:, 1]

best_threshold = 0.5
best_f1        = 0.0

thresholds = np.arange(0.1, 0.91, 0.01)
f1_scores  = []

for thresh in thresholds:
    preds = (y_val_proba >= thresh).astype(int)
    f1    = f1_score(y_val, preds, average='weighted', zero_division=0)
    f1_scores.append(f1)
    if f1 > best_f1:
        best_f1        = f1
        best_threshold = thresh

print(f"✅ Best threshold : {best_threshold:.2f}")
print(f"   Best weighted F1 on validation: {best_f1:.4f}")

# Show F1 at a few thresholds for reference
print("\nF1 at selected thresholds:")
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (y_val_proba >= t).astype(int)
    f = f1_score(y_val, p, average='weighted', zero_division=0)
    marker = " ← BEST" if abs(t - best_threshold) < 0.015 else ""
    print(f"  threshold={t:.1f}  →  weighted F1 = {f:.4f}{marker}")


# ─────────────────────────────────────────────
# PHASE 6 — GENERATE FINAL submission.csv
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("PHASE 6: Generating final submission.csv...")
print("="*60)

# Get probabilities on the FULL dataset
full_proba = xgb_model.predict_proba(X_scaled)[:, 1]

# Apply the best threshold we found
final_predictions = (full_proba >= best_threshold).astype(int)

# Build submission dataframe
submission = pd.DataFrame({
    'time'      : features['time'].values,
    'spoofed'   : final_predictions,
    'confidence': full_proba.round(4)
})

# Sanity check — ensure all times from submission_format are present
assert len(submission) == len(submission_format), \
    f"Row count mismatch! {len(submission)} vs {len(submission_format)}"
assert list(submission['time']) == list(submission_format['time']), \
    "Time order mismatch!"

# Save
submission.to_csv("submission.csv", index=False)

print(f"\n✅ submission.csv saved successfully!")
print(f"\n── Final predictions summary ──")
print(f"   Normal  (0): {(submission['spoofed']==0).sum():,}")
print(f"   Spoofed (1): {(submission['spoofed']==1).sum():,}")
print(f"\nFirst 10 rows of submission:\n")
print(submission.head(10).to_string(index=False))

print("\n" + "="*60)
print("✅ PIPELINE COMPLETE! Submit 'submission.csv'")
print("="*60)


# ─────────────────────────────────────────────
# BONUS: Feature Importance (for README/report)
# ─────────────────────────────────────────────
print("\n── Top 15 most important features (for your README) ──")
importances = pd.DataFrame({
    'feature'   : feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print(importances.head(15).to_string(index=False))
print("\n(These are the features your model relies on most for spoofing detection)")

✅ Best threshold : 0.62
   Best weighted F1 on validation: 0.9875

F1 at selected thresholds:
  threshold=0.3  →  weighted F1 = 0.9844
  threshold=0.4  →  weighted F1 = 0.9868
  threshold=0.5  →  weighted F1 = 0.9871
  threshold=0.6  →  weighted F1 = 0.9872
  threshold=0.7  →  weighted F1 = 0.9861

PHASE 6: Generating final submission.csv...

✅ submission.csv saved successfully!

── Final predictions summary ──
   Normal  (0): 40,593
   Spoofed (1): 7,151

First 10 rows of submission:

  time  spoofed  confidence
111402        0      0.0001
111403        0      0.0000
111404        0      0.0001
111405        0      0.0004
111406        0      0.0001
111407        0      0.0000
111408        0      0.0000
111409        0      0.0000
111410        0      0.0000
111411        0      0.0002

✅ PIPELINE COMPLETE! Submit 'submission.csv'

── Top 15 most important features (for your README) ──
                 feature  importance
         active_channels    0.180991
        doppler_roll_std 